# TrappedAtomsSimulation_RejectionSamplingTest

---



Author: Paul Christ

Date: 04.11.2025


In [1]:
# Import
from TrappedAtomsSimulation.initialization import initialize_one_temp_gaussian_state, initialize_two_temp_gaussian_state
from TrappedAtomsSimulation.force_calculation import calculate_interaction_strength
from TrappedAtomsSimulation.plot_utils import plot_energy_and_error, plot_thermalization, plot_potential_and_distribution_by_axis
from TrappedAtomsSimulation.trap_potential_extended import calculate_single_beam_X_AXIS, calculate_single_beam_Y_AXIS, calculate_crossed_beam_dipole_potential, calculate_U0, calculate_crossed_trap_frequencies
from TrappedAtomsSimulation.initialization_rejection_sampling import prepare_one_temp_simulation_rejection, prepare_two_temp_simulation_rejection

import torch
import math
import time

In [2]:
### troch settings
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dtype = torch.float32
kB = 1.380649e-23  # J/K
c_light = 299_792_458.0 # m/s
pi = math.pi

## Trap

In [4]:
trap_frequencies_Hz = (1000.0, 1050.0, 1200.0)
# Atommasse 87Rb
m_rb87 = 1.44160648e-25  # kg

# D1 Linie
omega_0_D1 = 2 * pi * 377.1074635e12 # rad/s
Gamma_D1 = 2 * pi * 5.746e6         # rad/s

# D2 Linie
omega_0_D2 = 2 * pi * 384.230484e12 # rad/s
Gamma_D2 = 2 * pi * 6.065e6         # rad/s

# Laserfrequenz
omega_L = 2 * pi * 280.179867e12    # rad/s


# Strahl 1 (X-Achse)
P_x = 5.18      # W
w0_x_SI = 41e-6   # m (41 µm)
zR_x_SI = 4.94e-3 # m (4.94 mm)

# Strahl 2 (Y-Achse)
P_y = 6.52      # W
w0_y_SI = 46e-6   # m (46 µm)
zR_y_SI = 6.21e-3 # m (6.21 mm)


# Gemeinsame Atom/Laser-Parameter
atom_laser_params = {
    "omega_L": omega_L,
    "omega_0_D1": omega_0_D1, "Gamma_D1": Gamma_D1,
    "omega_0_D2": omega_0_D2, "Gamma_D2": Gamma_D2
}


# Berechne U0 für Strahl 1
U0_1_SI = calculate_U0(P=P_x, w0=w0_x_SI, **atom_laser_params)

# Berechne U0 für Strahl 2
U0_2_SI = calculate_U0(P=P_y, w0=w0_y_SI, **atom_laser_params)

U0_1_in_uK = (U0_1_SI / kB) * 1e6
U0_2_in_uK = (U0_2_SI / kB) * 1e6

print("--- Berechnete Potentialtiefen (U0) ---")
print(f"  Strahl 1: {U0_1_in_uK:.1f} µK ")
print(f"  Strahl 2: {U0_2_in_uK:.1f} µK ")

trap_frequencies = calculate_crossed_trap_frequencies(
    U0_1=U0_1_SI, w0_1=w0_x_SI, zR_1=zR_x_SI,
    U0_2=U0_2_SI, w0_2=w0_y_SI, zR_2=zR_y_SI,
    m=m_rb87
)

print("\n--- Berechnete Frequenzen ---")
print(f"  f_x: {trap_frequencies['freq_x_hz']:.0f} Hz")
print(f"  f_y: {trap_frequencies['freq_y_hz']:.0f} Hz")
print(f"  f_z: {trap_frequencies['freq_z_hz']:.0f} Hz")
print(f"  f_mean: {trap_frequencies['freq_mean_hz']:.0f} Hz")


--- Berechnete Potentialtiefen (U0) ---
  Strahl 1: -294.8 µK 
  Strahl 2: -294.8 µK 

--- Berechnete Frequenzen ---
  f_x: 1163 Hz
  f_y: 1305 Hz
  f_z: 1747 Hz
  f_mean: 1384 Hz


### One group - Rejection Sampling

In [5]:
N_PARTICLES = 20000          
TEMPERATURE_K = 1e-6       
SIMULATION_TIME_S = 0.001      ### hier eigentlich unnötig, da nur Initialisierung getestet wird
TIMESTEP_S = 1e-5             
SUBSTEPS = 5

 
# Wird für die Initialisierung der Gauß-verteilung un dsampliung bounding box benötigt
TRAP_FREQUENCIES_HZ = (trap_frequencies['freq_x_hz'],trap_frequencies['freq_y_hz'],trap_frequencies['freq_z_hz'])

# --- 3. Physikalische Wechselwirkungsparameter ---

r0_phys,C_phys = calculate_interaction_strength(10)

 # --- 4. Initialisierung ---
 
sim_init_params = initialize_one_temp_gaussian_state(
    n_particles=N_PARTICLES,
    temp_k=TEMPERATURE_K,
    omega_phys_hz=TRAP_FREQUENCIES_HZ,  
    t_end_s=SIMULATION_TIME_S,
    dt_s=TIMESTEP_S,
    r0_phys=r0_phys,
    C_phys=C_phys,
    precision=torch.float32,
    device=device
)

# Skalierungsfaktoren  
L0 = sim_init_params['L0_m']
E0 = sim_init_params['E0_J']
T0 = sim_init_params['T0_s']

 

# Parameter für die Dipolfalle
trap_params_extended = {
    "P_x": P_x, "P_y": P_y,
    "w0_x": w0_x_SI, "w0_y": w0_y_SI,
    "s0_x": zR_x_SI, "s0_y": zR_y_SI,
    "omega_L": omega_L,
    "omega_0_D1": omega_0_D1, "Gamma_D1": Gamma_D1,
    "omega_0_D2": omega_0_D2, "Gamma_D2": Gamma_D2,
    "L0": L0,
    "E0": E0
}

sim_init_params_rejection_sampling = prepare_one_temp_simulation_rejection(
    n_particles =N_PARTICLES,
    temp_k=TEMPERATURE_K,
    omega_phys_hz=TRAP_FREQUENCIES_HZ,
    t_end_s=SIMULATION_TIME_S,
    dt_s=TIMESTEP_S,
    r0_phys=r0_phys,
    C_phys=C_phys,
    trap_force_func=calculate_crossed_beam_dipole_potential,
    trap_params=trap_params_extended,
    precision=torch.float32,
    device=device
)



plot_potential_and_distribution_by_axis(
        positions=sim_init_params['q0'],
        trap_force_func=calculate_crossed_beam_dipole_potential,
        trap_params=trap_params_extended,
        title_suffix="Gauss function with approx. frequencies (t=0)",
        bins=100
    )
plot_potential_and_distribution_by_axis(
        positions=sim_init_params_rejection_sampling['q0'],
        trap_force_func=calculate_crossed_beam_dipole_potential,
        trap_params=trap_params_extended,
        title_suffix="Rejection Sampling (t=0)",
        bins=100
    )

--- Skalierung basierend auf T = 1.00e-06 K ---
Längenskala L0: 1.34e-06 m, Energieskala E0: 1.38e-29 J, Zeitskala T0: 1.37e-04 s
Initialisiere 20000 Teilchen bei T = 1.00e-06 K
--- Initialisierung abgeschlossen ---

--- Skalierung basierend auf T_ref = 1.00e-06 K ---
Längenskala L0: 1.34e-06 m, Energieskala E0: 1.38e-29 J...
Verwende Bounding Box basierend auf T = 1.00e-06 K.
Initialisiere Gruppe (N=20000, T=1.00e-06 K)...
  -> Starte Rejection Sampling für 20000 Teilchen (T/T_ref = 1.00)...


InductorError: RuntimeError: 0 active drivers ([]). There should only be one.

Set TORCHDYNAMO_VERBOSE=1 for the internal stack trace (please do this especially if you're reporting a bug to PyTorch). For even more developer context, set TORCH_LOGS="+dynamo"
